In [1]:
import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import VCF_parser

In [ ]:
import time
import re
import gzip
import copy
import builtins
import pandas as pd

# BASIC READING FUNCTIONS

class VCFMetaEntry(dict):
    """
    Class for processing VCF metadata entries so they can be used for parsing the main file contents
    """
    
    meta_id = None
    number = None
    desc = None
    type = None

    
    def __init__(self, field):
        self.field = field


def get_read_params(file_path, force_format='infer'):
    """
    Checks the supplied path for file extensions and returns a tibble containing:
    - the module from which .open function will be called (may change it later)
    - the read mode string for that .open function
    """
    
    if force_format == 'vcf':
        return builtins, 'r'

    is_gzip = re.search(r'\.(gz|gzip)$', file_path)

    if is_gzip:
        vcf_file_handler_mod = gzip
        readmode = 'rt'
    else:
        vcf_file_handler_mod = builtins
        readmode = 'r'

    return vcf_file_handler_mod, readmode


def read_header_line(line):
    """
    Reads an unprocessed line from VCF header and returns a VCFHeaderEntry object
    """

    try:
        entry = VCFMetaEntry(re.search(r'##[A-Z]+=', line).group().replace("##", "").replace("=", ""))
        entry.meta_id = re.search(r'ID=([A-Z]|[a-z]|[0-9]|_)+', line).group().replace("ID=", "")
    except AttributeError:
        return None

    try:
        number_val = int(re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", ""))
        entry.number = number_val
    except ValueError:
        number_val = re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", "")
        entry.number = number_val
    except AttributeError:
        entry.number = None

    try:
        type_val = re.search(r'Type=([A-Z]|[a-z]|[0-9])+,', line).group().replace("Type=", "").replace(",", "")
        entry.type = type_val
    except AttributeError:
        entry.type = None

    try:
        entry.desc = re.search(r'Description=".+"', line).group().replace('Description=\"', '').replace('"', '')
    except AttributeError:
        entry.desc = None
    
    return entry

def process_meta_entry(vcf_dict, meta_dict, meta_entry):
    """
    Accepts a VCFMetaEntry object along with 2 dictionaries \
    - first dict is to be used for storing the full vcf data
    - second dict is to be used for storing vcf metadata

    Updates the dictionaries with information from the meta entry
    - adds an {entry field}_{entry id}: [] item to the vcf data dict
    - adds an {entry field}_{entry id}: (entry number, entry value type, entry description)\
    item to the meta dict. Note that meta dict is a nested dictionary \
    (it is a dict with FILTER, FORMAT, and INFO fields, all of which are used\
    to address a dictionary with corresponding meta entry IDs as keys), 
    """
    if meta_entry is not None:
        if meta_entry.field in meta_dict:
            meta_dict[meta_entry.field].update(
                {
                    f'{meta_entry.field}_{meta_entry.meta_id}':\
                    (meta_entry.number, meta_entry.type, meta_entry.desc)
                }
            )
            # print(f'[{meta_entry.field}] {meta_entry.meta_id}, number: {meta_entry.number}, type: {meta_entry.type}, desc: {meta_entry.desc}')
            vcf_dict.update({f'{meta_entry.field}_{meta_entry.meta_id}': []})


def process_filter_entry(vcf_dict, meta_dict, filter_str, filter_delim=';'):
    """
    Accepts a vcf dict and meta dict along with a filter entry string.
    Iterates over FILTER entries in the meta dict.
    If any of the FILTER_{filter entry id} appear in the string, \
    appends the corresponding list in the vcf dict with 1
    Otherwise, appends the corresponding list with 0
    """
    filter_entry_with_prefix = [f'FILTER_{x}' for x in filter_str.split(filter_delim)]
    # print(filter_entry_with_prefix)
    
    for filter_field in meta_dict['FILTER']:
        if filter_field in filter_entry_with_prefix:
            vcf_dict[filter_field].append(1)
        else:
            vcf_dict[filter_field].append(0)


def process_info_entry(vcf_dict, meta_dict, info_str, info_delim=';', info_eq='='):
    """
    Accepts a vcf dict and meta dict along with an info entry string.
    NOTE TO SELF - COULD PROB OPTIMIZE

    Splits the info string based on the supplied delimeter and equal sign
    Iterates over the resulting list and appends lists in the vcf dict with corresponding values
    If any field from vcf meta is not present in the string, \
    appends corresponding list in vcf dict with 0
    """

    info_dict = {}
    for info_item_raw in info_str.split(info_delim):
        info_item = info_item_raw.split(info_eq)
        if len(info_item) == 1:
            info_dict.update({f'INFO_{info_item[0]}': 1})
        elif len(info_item) == 2:
            info_dict.update({f'INFO_{info_item[0]}': info_item[1]})
        else:
            raise IndexError('INFO field entry with wrong syntaxis detected')
    
    info_fields = info_dict.keys()
    for info_field in meta_dict['INFO']:
        if info_field in info_fields:
            vcf_dict[info_field].append(info_dict[info_field])
        else:
            vcf_dict[info_field].append(0)


def process_genotype_entry(vcf_dict, meta_dict, gt_list, format_delim=':'):
    """
    Accepts a vcf dict and meta dict. Unlike other similar function, accepts genotype info \
    as a list of strings.

    Currently only supports a single genotype (FORMAT str + 1 genotype str)

    Iterates over the format fields and appends values to the corresponding vcf dict lists
    """
    format_dict = {
        f'FORMAT_{x[0]}': x[1] for x in
        zip(gt_list.pop(0).split(format_delim), gt_list.pop(0).split(format_delim))
    }

    format_fields = format_dict.keys()
    for format_field in meta_dict['FORMAT']:
        if format_field in format_fields:
            vcf_dict[format_field].append(format_dict[format_field])


def parse_vcf(
    vcf_path,
    filter_delim=';',
    info_delim=';', info_eq='=',
    format_present='infer', format_delim=':',
    line_limit=0, report_freq=100000,
    output_meta=False
):
    """
    
    """
    
    meta_dict = {
        'FILTER': {},
        'INFO': {},
        'FORMAT': {}
    }
    
    vcf_data = {
        'CHROM': [],
        'POS': [],
        'ID': [],
        'REF': [],
        'ALT': [],
        'QUAL': []
    }

    vcf_file_handler_mod, readmode = get_read_params(vcf_path, force_format=format_present)
    
    header_nlines = 0
    with vcf_file_handler_mod.open(vcf_path, readmode) as vcf:
        
        start_time = time.time()
        i = 0
        for i, line in enumerate(vcf):
            if i % report_freq == 0:
                current_time = time.time()
                print(f'Processed {i} entries \
                Elapsed time: {current_time - start_time} seconds.')
            
            # There are 2 major blocks processed differently: the header and the body
            # Currently the algorithm to process a headerless vcf file is not done, so a header is mandatory
    
            if line.startswith('#'):
                header_nlines += 1
                # Both the metadata and the header itself has a # / ## in the beginning
                if line.startswith('##'):
                    # print(line)
                    meta_entry = read_header_line(line)
                    process_meta_entry(vcf_data, meta_dict, meta_entry)
                else:
                    continue        
            else:
                if (i - header_nlines >= line_limit) & (line_limit != 0):
                    # print(vcf_data)
                    # print(meta_dict)
                    break
                
                line_list = line.split('\t')
                # print(line_list)
    
                # First fields are easily processed
                vcf_data['CHROM'].append(line_list.pop(0))
                vcf_data['POS'].append(line_list.pop(0))
                vcf_data['ID'].append(line_list.pop(0))
                vcf_data['REF'].append(line_list.pop(0))
                vcf_data['ALT'].append(line_list.pop(0))
                vcf_data['QUAL'].append(line_list.pop(0))
    
                # FILTER
                # The filter field is referenced to all possible filter field in the metadata
                # If an entry is present, then the column is saved with value 1, else - with 0
                process_filter_entry(
                    vcf_data, meta_dict,
                    filter_str=line_list.pop(0),
                    filter_delim=filter_delim
                )
    
                # INFO
                # Here the program iterates over the possible INFO field values listed in the header too
                # info_entry_with_prefix_raw = [f'INFO_{x}' for x in line_list.pop(0).split(info_delim)]
                process_info_entry(
                    vcf_data, meta_dict,
                    info_str=line_list.pop(0),
                    info_delim=info_delim, info_eq=info_eq
                )

                # FORMAT and genotype columns
                # Currently only a single genotype column is supported
                if line_list == []:
                    continue

                process_genotype_entry(
                    vcf_data, meta_dict,
                    line_list,
                    format_delim=format_delim
                )
                
    
    keys_for_deletion = []
    for vcf_data_key, vcf_data_item in vcf_data.items():
        if len(vcf_data_item) == 0:
            keys_for_deletion.append(vcf_data_key)
    if len(keys_for_deletion) > 0:
        print(f'Warning: items {", ".join(keys_for_deletion)} present in header but not present in data.')
    for key_for_deletion in keys_for_deletion:
        del vcf_data[key_for_deletion]
        # Small fix so an entire section does not get deleted if INFO/FORMAT/FILTER
        # is mentioned in the metadata
        if key_for_deletion in ['INFO', 'FORMAT', 'FILTER']:
            continue
        try:
            del meta_dict[re.match(r'(INFO)|(FORMAT)|(FILTER)', key_for_deletion).group()][key_for_deletion]
        except KeyError:
            print(f'Unexpected meta ID: {key_for_deletion}')
        # try:
        #     del meta_dict['INFO'][key_for_deletion]
        #     del meta_dict['FORMAT'][key_for_deletion]
        #     del meta_dict['FILTER'][key_for_deletion]
        # except KeyError:
        #     continue

    vcf_data = pd.DataFrame(vcf_data)

    vcf_data.POS = pd.to_numeric(vcf_data.POS)
    vcf_data.QUAL = pd.to_numeric(vcf_data.QUAL, errors='coerce')
    
    current_time = time.time()
    print(f'Finished processing. Processed {i - header_nlines} entries out of {len(vcf_data)}. Elapsed time: {current_time - start_time} seconds.')

    if output_meta:
        return vcf_data, meta_dict
        
    return vcf_data


# POST-PROCESSING FUNCTIONS

def combine_meta_dict(meta_dict):
    """
    """
    return {**meta_dict['FILTER'], **meta_dict['INFO'], **meta_dict['FORMAT']}

def split_meta_dict(meta_dict_combined):
    """
    """
    meta_dict = {
        'FILTER': {},
        'INFO': {},
        'FORMAT': {}
    }
    for meta_key, meta_item in meta_dict_combined.items():
        meta_dict[re.match(r'(INFO)|(FORMAT)|(FILTER)', meta_key).group()][meta_key] = meta_item
    
    return meta_dict

def get_split_meta_entry(meta_id, meta_info, values, output_type='Flag'):
    """
    """
    new_ids = [f'{meta_id}_{value}' for value in values]

    return {new_id: (1, output_type, f'SPLIT FROM {meta_id}, OLD DESC: {meta_info[2]}') for new_id in new_ids}
    

def get_processed_algorithm_df_and_meta(vcf_df, meta_dict, delim=',', alg_col_name='INFO_ALGORITHMS'):
    """
    
    """
    # algs = vcf_df[alg_col].str.split(delim).astype(set)
    alg_col = vcf_df[alg_col_name]
    alg_set = set()
    
    for uniq_alg_comb in alg_col.unique():
        alg_set = alg_set | set(uniq_alg_comb.split(delim))
        
    alg_dict = {f'{alg_col_name}_{key}': [0] * len(alg_col) for key in alg_set}
    
    for i, alg_item in enumerate(alg_col):
        for alg in alg_item.split(delim):
            alg_dict[f'{alg_col_name}_{alg}'][i] = 1

    meta_dict_copy = copy.deepcopy(meta_dict)
    old_alg_entry = meta_dict_copy['INFO'].pop(alg_col_name)
    new_alg_entry = get_split_meta_entry(alg_col_name, old_alg_entry, alg_set)
    # print(old_alg_entry, new_alg_entry)

    # print(pd.concat([vcf_df[vcf_df.columns.drop([alg_col_name])], pd.DataFrame(alg_dict)], axis=1))
    meta_dict_copy['INFO'].update(new_alg_entry)
    
    return pd.concat([vcf_df[vcf_df.columns.drop([alg_col_name])], pd.DataFrame(alg_dict)], axis=1), meta_dict_copy
        

def convert_col_types(vcf_df, meta_dict, types_from='meta'):
    """
    
    """
    if types_from == 'meta':
        type_convert_dict = {
            'String': str,
            'Integer': int,
            'Float': float,
            'Flag': int
        }
        for meta_id, meta_info in (meta_dict['INFO'] | meta_dict['FORMAT']).items():
            vcf_df[meta_id] = vcf_df[meta_id].astype(type_convert_dict[meta_info[1]])


def delete_mult_val_cols(vcf_df, meta_dict, exceptions=['INFO_ALGORITHMS'], vocal=False):
    """
    
    """
    if exceptions is None:
        exceptions = []
    
    keys_for_deletion = []
    for meta_group in meta_dict.values():
        for meta_id, meta_info in meta_group.items():
            # print(meta_id, meta_info)
            if meta_info[0] not in ['1', 1] + exceptions:
                keys_for_deletion.append(meta_id)

    if vocal:
        print(f'Dropping entries {to_drop} since they contain multiple values')

    for key_for_deletion in keys_for_deletion:
        del meta_dict[re.match(r'(INFO)|(FORMAT)|(FILTER)', key_for_deletion).group()][key_for_deletion]
    
    vcf_df.drop(keys_for_deletion, axis=1, inplace=True)

In [2]:
# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
gridss_path = f'{wd}/data/test_gridss_output.NA12878.vcf'
gnomad_path = '/home/users/allstaff/shikov.v/projects/SVensemble2/data/gnomad/4.1.0/downloads/gnomad.v4.1.sv.sites.vcf.gz'

In [3]:
test_mod, test_mode = VCF_parser.get_read_params(gridss_path)

In [3]:
giab_vcf = VCF_parser.parse_vcf('/vast/scratch/users/shikov.v/SVensemble2/data/NA12878/downloads/HG001_GRCh38_1_22_v4.2.1_benchmark.vcf')
# giab_vcf = pd.read_csv('/vast/scratch/users/shikov.v/SVensemble2/data/NA12878/downloads/HG001_GRCh38_1_22_v4.2.1_benchmark.vcf', delimiter='\t')

Processed 0 entries                 Elapsed time: 2.3365020751953125e-05 seconds.
Processed 100000 entries                 Elapsed time: 1.1482651233673096 seconds.
Processed 200000 entries                 Elapsed time: 2.2542383670806885 seconds.
Processed 300000 entries                 Elapsed time: 3.3792943954467773 seconds.
Processed 400000 entries                 Elapsed time: 4.4742841720581055 seconds.
Processed 500000 entries                 Elapsed time: 5.582178831100464 seconds.
Processed 600000 entries                 Elapsed time: 6.672621726989746 seconds.
Processed 700000 entries                 Elapsed time: 7.742103338241577 seconds.
Processed 800000 entries                 Elapsed time: 8.821240186691284 seconds.
Processed 900000 entries                 Elapsed time: 9.887232542037964 seconds.
Processed 1000000 entries                 Elapsed time: 10.967844724655151 seconds.
Processed 1100000 entries                 Elapsed time: 12.022504806518555 seconds.
Processe

ValueError: All arrays must be of the same length

In [4]:
test_mod

<module 'builtins' (built-in)>

In [5]:
def split_vcf_by_chrom(vcf_df, chrom_list=None):
    if chrom_list is not None:
        return {chrom: vcf_df.loc[vcf_df.CHROM == chrom] \
                for chrom in chrom_list}
    return {chrom: gnomad_vcf_df.loc[gnomad_vcf_df.CHROM == chrom] \
                for chrom in pd.unique(vcf_df.CHROM)}

In [6]:
def generate_truth_list(vcf_df, ref_df_dict, report_freq=100000):

    truth_list = []
    pos_series = vcf_df.POS
    chrom_series = vcf_df.CHROM
    
    if set(ref_df_dict.keys()) != set(chrom_series.unique()):
        raise KeyError('Chromosomes in prediction dataset do not match chromosomes in gnomad')
    
    for i, (pos, chrom) in enumerate(zip(pos_series, chrom_series)):
        # print(f'{chrom}: {pos}')
        # overlapping_ref_positions = (gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)
    
        truth_list.append(
            ((ref_df_dict[chrom].POS > pos - 25) & (ref_df_dict[chrom].POS < pos + 25)).any()
        )
        
        if i % report_freq == 0:
            print(f'processed {i} entries')

    return truth_list

In [7]:
gnomad_vcf_df = VCF_parser.parse_vcf(gnomad_path, line_limit=500000)

Processed 0 entries                 Elapsed time: 0.030124187469482422 seconds.
Processed 100000 entries                 Elapsed time: 34.149534463882446 seconds.
Processed 200000 entries                 Elapsed time: 69.98905372619629 seconds.
Processed 300000 entries                 Elapsed time: 105.60838150978088 seconds.
Processed 400000 entries                 Elapsed time: 140.9914288520813 seconds.
Processed 500000 entries                 Elapsed time: 176.18842148780823 seconds.
Finished processing. Processed 500000 entries out of 500000. Elapsed time: 239.9264633655548 seconds.


In [8]:
gridss_vcf_df = VCF_parser.parse_vcf(gridss_path)

Processed 0 entries                 Elapsed time: 0.009137868881225586 seconds.
Processed 100000 entries                 Elapsed time: 5.46272611618042 seconds.
Processed 200000 entries                 Elapsed time: 10.919637441635132 seconds.
Processed 300000 entries                 Elapsed time: 16.406007289886475 seconds.
Processed 400000 entries                 Elapsed time: 21.87128496170044 seconds.
Processed 500000 entries                 Elapsed time: 27.40699052810669 seconds.
Processed 600000 entries                 Elapsed time: 33.01365089416504 seconds.
Processed 700000 entries                 Elapsed time: 38.53498554229736 seconds.
Processed 800000 entries                 Elapsed time: 44.06725788116455 seconds.
Processed 900000 entries                 Elapsed time: 49.60082411766052 seconds.
Processed 1000000 entries                 Elapsed time: 55.10003733634949 seconds.
Finished processing. Processed 1007409 entries out of 1007410. Elapsed time: 81.83646869659424 sec

In [59]:
gnomad_vcf_df = split_vcf_by_chrom(gnomad_vcf_df, pd.unique(gridss_vcf_df.CHROM))

In [64]:
truth_list = generate_truth_list(gridss_vcf_df, gnomad_vcf_df)
gridss_vcf_df['MATCH_FOUND'] = truth_list

processed 0 entries
processed 100000 entries
processed 200000 entries
processed 300000 entries
processed 400000 entries
processed 500000 entries
processed 600000 entries
processed 700000 entries
processed 800000 entries
processed 900000 entries
processed 1000000 entries


In [66]:
gnomad_vcf_df = pd.concat(gnomad_vcf_df, ignore_index=True)
# gridss_vcf_df = pd.concat(gridss_vcf_df, ignore_index=True)

In [67]:
gnomad_vcf_df = gnomad_vcf_df.sort_values(by=['POS'])
gridss_vcf_df = gridss_vcf_df.sort_values(by=['POS'])

In [31]:
gridss_vcf_df = pd.merge_asof(gridss_vcf_df, gnomad_vcf_df, on='POS', by='CHROM', direction='nearest')

In [49]:
gridss_vcf_df.iloc[0:400,:].to_csv('~/projects/SVensemble2/truth_set_gen/saved/test_dataset.tsv', sep='\t')

In [55]:
test_df = gridss_vcf_df.iloc[0:400,:]

In [56]:
test_df = pd.concat([test_df[['CHROM', 'ID_x', 'REF_x']], test_df.select_dtypes(include=np.number), test_df[['MATCH_FOUND']]], axis=1)

In [57]:
test_df.to_csv('~/projects/SVensemble2/truth_set_gen/saved/test_dataset.filtered.tsv', sep='\t')

In [2]:
#!/home/users/allstaff/shikov.v/.conda/envs/SVEnsemble2/bin/python

#SBATCH --job-name=truth_set_gen
#SBATCH --mem=100GB
#SBATCH --cpus-per-task=2
#SBATCH --time=24:00:00

import os
import sys
sys.path.append('/home/users/allstaff/shikov.v/vast/SVensemble2/truth_set_gen/scripts')

import time
import re
import copy
import numpy as np
import pandas as pd
import matplotlib as mpl
import VCF_parser

def split_vcf_by_chrom(vcf_df, chrom_list=None):
    if chrom_list is not None:
        return {chrom: vcf_df.loc[vcf_df.CHROM == chrom] \
                for chrom in chrom_list}
    return {chrom: gnomad_vcf_df.loc[gnomad_vcf_df.CHROM == chrom] \
                for chrom in pd.unique(vcf_df.CHROM)}


def generate_truth_list(vcf_df, ref_df_dict, pos_interval=25, report_freq=100000):

    truth_list = []
    pos_series = vcf_df.POS
    chrom_series = vcf_df.CHROM
    
    if set(ref_df_dict.keys()) != set(chrom_series.unique()):
        raise KeyError('Chromosomes in prediction dataset do not match chromosomes in gnomad')
    
    for i, (pos, chrom) in enumerate(zip(pos_series, chrom_series)):
        # print(f'{chrom}: {pos}')
        # overlapping_ref_positions = (gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)
    
        truth_list.append(
            ((ref_df_dict[chrom].POS > pos - pos_interval) & (ref_df_dict[chrom].POS < pos + pos_interval)).any()
        )
        
        if i % report_freq == 0:
            print(f'processed {i} entries')

    return truth_list


# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
gridss_path = f'{wd}/data/test_gridss_output.NA12878.vcf'
gnomad_path = '/home/users/allstaff/shikov.v/projects/SVensemble2/data/gnomad/4.1.0/downloads/gnomad.v4.1.sv.sites.vcf.gz'

gnomad_vcf_df, gnomad_meta = VCF_parser.parse_vcf(gnomad_path, line_limit=100000, output_meta=True)
gridss_vcf_df, gridss_meta = VCF_parser.parse_vcf(gridss_path, line_limit=100000, output_meta=True)


Processed 0 entries                 Elapsed time: 0.0003998279571533203 seconds.
Processed 100000 entries                 Elapsed time: 25.533762216567993 seconds.
Finished processing. Processed 100000 entries out of 100000. Elapsed time: 32.18179798126221 seconds.
Processed 0 entries                 Elapsed time: 2.3603439331054688e-05 seconds.
Processed 100000 entries                 Elapsed time: 4.241633176803589 seconds.
Finished processing. Processed 100000 entries out of 100000. Elapsed time: 5.8244853019714355 seconds.


In [3]:
# gnomad_df_copy = gnomad_df.copy(deep=True)
# gnomad_meta_copy = copy.deepcopy(gnomad_meta)
gnomad_vcf_df, gnomad_meta = VCF_parser.get_processed_algorithm_df_and_meta(gnomad_vcf_df, gnomad_meta)
VCF_parser.delete_mult_val_cols(gnomad_vcf_df, gnomad_meta)
VCF_parser.convert_col_types(gnomad_vcf_df, gnomad_meta)


In [4]:

VCF_parser.delete_mult_val_cols(gridss_vcf_df, gridss_meta)
VCF_parser.convert_col_types(gridss_vcf_df, gridss_meta)


In [5]:

gnomad_vcf_df = split_vcf_by_chrom(gnomad_vcf_df, pd.unique(gridss_vcf_df.CHROM))

truth_list = generate_truth_list(gridss_vcf_df, gnomad_vcf_df)
gridss_vcf_df['MATCH_FOUND'] = truth_list

gnomad_vcf_df = pd.concat(gnomad_vcf_df, ignore_index=True)

gnomad_vcf_df = gnomad_vcf_df.sort_values(by=['POS'])
gridss_vcf_df = gridss_vcf_df.sort_values(by=['POS'])

truth_suffix = '_truth_set'
gnomad_vcf_df = gnomad_vcf_df.add_suffix(truth_suffix)


processed 0 entries


In [6]:
gnomad_vcf_df

,CHROM_truth_set,POS_truth_set,ID_truth_set,REF_truth_set,ALT_truth_set,QUAL_truth_set,INFO_CHR2_truth_set,INFO_CPX_TYPE_truth_set,INFO_END_truth_set,INFO_END2_truth_set,...,INFO_FREQ_HOMALT_sas_XX_truth_set,INFO_CN_NUMBER_sas_XX_truth_set,INFO_CN_NONREF_COUNT_sas_XX_truth_set,INFO_CN_NONREF_FREQ_sas_XX_truth_set,INFO_GRPMAX_AF_truth_set,INFO_ALGORITHMS_depth_truth_set,INFO_ALGORITHMS_rescan_truth_set,INFO_ALGORITHMS_manta_truth_set,INFO_ALGORITHMS_wham_truth_set,INFO_ALGORITHMS_melt_truth_set
0,chr1,10000,gnomAD-SV_v3_DUP_chr1_01c2781c,N,<DUP>,134.0,chr1,0,295666,0,...,0.000000,0,0,0.0,0.005325,1,0,0,0,0
1,chr1,10434,gnomAD-SV_v3_BND_chr1_1a45f73a,N,<BND>,260.0,chr2,0,10434,242183515,...,0.000000,0,0,0.0,0.168454,0,0,1,0,0
2,chr1,10440,gnomAD-SV_v3_BND_chr1_3fa36917,N,<BND>,198.0,chr3,0,10440,10977,...,0.000000,0,0,0.0,0.012825,0,0,1,0,0
3,chr1,10450,gnomAD-SV_v3_BND_chr1_933a2971,N,<BND>,287.0,chr18,0,10450,80258275,...,0.006667,0,0,0.0,0.366535,0,0,1,0,0
4,chr1,10464,gnomAD-SV_v3_BND_chr1_7bbf34b5,N,<BND>,198.0,chr12,0,10464,10502,...,0.000000,0,0,0.0,0.068683,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,chr1,117397686,gnomAD-SV_v3_INS_chr1_01e1bc49,N,<INS>,1.0,chr1,0,117397708,0,...,0.000000,0,0,0.0,0.007955,0,0,1,0,0
99996,chr1,117398374,gnomAD-SV_v3_DEL_chr1_5cdfc7ae,N,<DEL>,1.0,chr1,0,117399169,0,...,0.000000,0,0,0.0,0.500000,0,0,0,1,0
99997,chr1,117399046,gnomAD-SV_v3_INS_chr1_ff761ec8,N,<INS:ME:ALU>,140.0,chr1,0,117399097,0,...,0.000000,0,0,0.0,0.000080,0,0,0,0,1
99998,chr1,117409020,gnomAD-SV_v3_INS_chr1_606b3d63,N,<INS:ME:SVA>,898.0,chr1,0,117409056,0,...,0.000000,0,0,0.0,0.000228,0,0,0,0,1


In [7]:

gridss_vcf_df = pd.concat(
    [
        gridss_vcf_df.loc[~gridss_vcf_df['MATCH_FOUND']],
        pd.merge_asof(
            gridss_vcf_df.loc[gridss_vcf_df['MATCH_FOUND']],
            gnomad_vcf_df,
            left_on='POS', right_on=f'POS{truth_suffix}',
            left_by='CHROM', right_by=f'CHROM{truth_suffix}',
            direction='nearest'
        )
    ]
)



In [8]:
gridss_vcf_df

,CHROM,POS,ID,REF,ALT,QUAL,FORMAT_ANRP,FORMAT_ANRPQ,FORMAT_ANSR,FORMAT_ANSRQ,...,INFO_FREQ_HOMALT_sas_XX_truth_set,INFO_CN_NUMBER_sas_XX_truth_set,INFO_CN_NONREF_COUNT_sas_XX_truth_set,INFO_CN_NONREF_FREQ_sas_XX_truth_set,INFO_GRPMAX_AF_truth_set,INFO_ALGORITHMS_depth_truth_set,INFO_ALGORITHMS_rescan_truth_set,INFO_ALGORITHMS_manta_truth_set,INFO_ALGORITHMS_wham_truth_set,INFO_ALGORITHMS_melt_truth_set
0,chr1,9952,gridss0bf_1o,N,]chr1:10498]N,498.42,27,408.2,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,10317,gridss0fb_13o,A,ACCCCTTCCCC[chr1:10328[,241.38,0,0.0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,10325,gridss0fb_14o,C,CCAACCCC[chr1:10333[,230.11,0,0.0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,10328,gridss0fb_13h,A,]chr1:10317]CCCCTTCCCCA,241.38,0,0.0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,10332,gridss0f_58b,C,CTTAGCCCTTAGCCCTGGCCCTAGCCCTA.,153.86,0,0.0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8972,chr1,117309611,gridss11fb_5381o,A,A[chr1:117309632[,1263.23,0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.001289,0.0,0.0,0.0,1.0,0.0
8973,chr1,117309625,gridss11f_22166b,A,AAACAACAAGTATCTGAAATG.,336.88,0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.001289,0.0,0.0,0.0,1.0,0.0
8974,chr1,117309632,gridss11fb_5381h,A,]chr1:117309611]A,1263.23,0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.001289,0.0,0.0,0.0,1.0,0.0
8975,chr1,117397648,gridss11b_22385b,T,.TAAAATGTATTTTAAAATTTAAAAAGCATTTTTTTTGCCCCTCCT...,459.62,0,0.0,0,0.0,...,0.0,0.0,0.0,0.0,0.000246,0.0,0.0,1.0,0.0,0.0


In [9]:
cols_to_drop = [f'CHROM{truth_suffix}', f'POS{truth_suffix}', f'REF{truth_suffix}']
cols_to_drop

['CHROM_truth_set', 'POS_truth_set', 'REF_truth_set']

In [11]:
gridss_vcf_df = gridss_vcf_df[gridss_vcf_df.columns.drop(cols_to_drop)]



In [ ]:
gridss_vcf_df = pd.concat(
    [
        gridss_vcf_df[['CHROM', 'ID', 'REF']],
        gridss_vcf_df.select_dtypes(include=np.number),
        gridss_vcf_df[['MATCH_FOUND']],
        gridss_vcf_df.filter(regex=f'INFO_ALGORITHMS{truth_suffix}')
    ],
    axis=1
)

gridss_vcf_df.to_csv('~/projects/SVensemble2/truth_set_gen/saved/gridss_vcf_df.filtered.test.tsv', sep='\t')
